# Evidence-oriented clinical review (synthetic)
This offline demonstration creates synthetic M1→M4 inputs. Bands are review routing, not diagnosis, risk, pathogenicity, or actionability.

In [ ]:
import json
import shutil
import tempfile
from hashlib import sha256
from pathlib import Path

from genome_evidence.evidence import ingest_clinvar_vcv, link_external_evidence
from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run
from genome_evidence.prioritization import prioritize_clinical_variants
from genome_evidence.prioritization.models import AnalysisContext, ClinicalPrioritizationConfig

In [ ]:
root = Path(tempfile.mkdtemp(prefix="synthetic-m4-"))
source = root / "synthetic.txt"
source.write_text("# genome build: GRCh38\nsynthetic-evidence-marker\t1\t101\tAG\n")
markers = root / "markers.json"
markers.write_text(
    json.dumps(
        [
            {
                "marker_id": "synthetic-evidence-marker",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 101,
                "reference": "A",
                "alternate": "G",
                "orientation": "none",
                "orientation_authoritative": True,
            }
        ]
    )
)
fasta = root / "synthetic.fa"
fasta.write_text(">1\n" + "A" * 200 + "\n")
m1 = root / "m1"
ingest_23andme(source, m1, Ingest23andMeConfig(genome_build_override="GRCh38"))
m2 = root / "m2"
normalize_m1_run(m1, m2, NormalizationConfig(marker_definitions=markers, target_reference=fasta))
xml = root / "synthetic-clinvar.xml"
xml.write_text(
    '<ReleaseSet Dated="2026-07-01" ReleaseID="synthetic-m4">'
    '<VariationArchive Accession="VCV999000001" Version="1" RecordStatus="current">'
    '<ClassifiedRecord><SimpleAllele AlleleID="1"><SequenceLocation Assembly="GRCh38" '
    'Chr="1" positionVCF="101" referenceAlleleVCF="A" alternateAlleleVCF="G"/>'
    '</SimpleAllele><Classifications><GermlineClassification DateLastEvaluated="2025-01-01">'
    "<Description>Pathogenic</Description><ReviewStatus>criteria provided, single submitter"
    "</ReviewStatus></GermlineClassification></Classifications>"
    '<ClinicalAssertion Accession="SCV999000001" Version="1" RecordStatus="current">'
    '<Submitter Name="Fabricated Laboratory"/><GermlineClassification '
    'DateLastEvaluated="2025-01-01"><Description>Uncertain significance</Description>'
    "<ReviewStatus>criteria provided, single submitter</ReviewStatus>"
    "</GermlineClassification></ClinicalAssertion></ClassifiedRecord></VariationArchive>"
    "</ReleaseSet>"
)
evidence = root / "evidence"
ingest_clinvar_vcv(xml, evidence)
annotation = root / "annotation"
link_external_evidence(m2, evidence, annotation)
policy = root / "policy.json"
repo = Path(__import__("genome_evidence").__file__).parents[2]
shutil.copyfile(repo / "references/clinvar-germline-review-policy-v1.json", policy)
m4 = root / "m4"
result = prioritize_clinical_variants(
    m2,
    evidence,
    annotation,
    m4,
    ClinicalPrioritizationConfig(
        policy_path=policy, analysis_context=AnalysisContext.GERMLINE_CONSTITUTIONAL
    ),
)

In [ ]:
assert result.policy_identity.file_sha256 == sha256(policy.read_bytes()).hexdigest()
assert all(
    c.priority_band.value not in {"diagnosis", "risk", "actionability"} for c in result.candidates
)
assert len(result.candidates) == 1
assert result.candidates[0].priority_band.value == "review_first"
assert all(p.scv_assertion_ids and p.vcv_assertion_ids for p in result.profiles)
assert {link.source_term for link in result.candidate_assertion_links} == {
    "Pathogenic",
    "Uncertain significance",
}
assert all(link.assertion_instance_id for link in result.candidate_assertion_links)
assert all("diagnosis" not in rationale.explanation for rationale in result.rationales)
manifest = json.loads((m4 / "manifest.json").read_text())
assert all(
    sha256((m4 / name).read_bytes()).hexdigest() == digest
    for name, digest in manifest["artifacts"].items()
)
assert "not a negative genetic test" in (m4 / "prioritization_report.md").read_text()
shutil.rmtree(root)